## Лабораторная работа 11: Модуляция и дискретизация

### Упражнение 11.1: Антиалиасинговый фильтр

In [ ]:
import sys
sys.path.insert(0, '../ThinkDSP/code')

from thinkdsp import read_wave
from thinkdsp import decorate
import matplotlib.pyplot as plt
import numpy as np

### Проблема алиасинга

При дискретизации сигнала с недостаточной частотой возникает алиасинг - частоты выше частоты Найквиста "складываются" и становятся неотличимыми от низких частот. Антиалиасинговый фильтр применяется ДО дискретизации для предотвращения этого эффекта.

### Загрузка исходного сигнала

In [ ]:
# Загрузка барабанного соло
wave = read_wave('../ThinkDSP/code/263868__kevcio__amen-break-a-160-bpm.wav')
wave.normalize()

wave.plot()
plt.title('Исходный сигнал (44100 Гц)')
decorate(xlabel='Время (с)', ylabel='Амплитуда')
plt.show()

# Прослушивание
wave.make_audio()

### Спектр исходного сигнала

In [ ]:
spectrum = wave.make_spectrum()
spectrum.plot(high=10000)
plt.title('Спектр исходного сигнала')
decorate(xlabel='Частота (Гц)', ylabel='Амплитуда')
plt.show()

### Дискретизация без антиалиасингового фильтра

In [ ]:
# Понижение частоты дискретизации до 11025 Гц (в 4 раза)
# Частота Найквиста = 5512.5 Гц
factor = 4
sampled = wave.copy()
sampled.ys = sampled.ys[::factor]
sampled.framerate = sampled.framerate // factor

print(f'Новая частота дискретизации: {sampled.framerate} Гц')
print(f'Частота Найквиста: {sampled.framerate / 2} Гц')

# Спектр после дискретизации
sampled_spectrum = sampled.make_spectrum()
sampled_spectrum.plot(high=10000)
plt.title('Спектр после дискретизации (без антиалиасинга)')
decorate(xlabel='Частота (Гц)', ylabel='Амплитуда')
plt.show()

# Прослушивание
sampled.make_audio()

**Комментарий:** Видны артефакты алиасинга - высокие частоты "отразились" в низкочастотную область.

### Применение антиалиасингового фильтра

In [ ]:
# Применяем НЧ фильтр ПЕРЕД дискретизацией
# Частота среза = частота Найквиста нового сигнала
cutoff = sampled.framerate / 2

# Фильтрация исходного сигнала
filtered_spectrum = wave.make_spectrum()
filtered_spectrum.low_pass(cutoff)
filtered_wave = filtered_spectrum.make_wave()

# Визуализация отфильтрованного спектра
filtered_spectrum.plot(high=10000)
plt.axvline(cutoff, color='r', linestyle='--', label=f'Частота среза: {cutoff} Гц')
plt.title('Спектр после антиалиасингового фильтра')
plt.legend()
decorate(xlabel='Частота (Гц)', ylabel='Амплитуда')
plt.show()

### Дискретизация с антиалиасинговым фильтром

In [ ]:
# Дискретизация отфильтрованного сигнала
sampled_filtered = filtered_wave.copy()
sampled_filtered.ys = sampled_filtered.ys[::factor]
sampled_filtered.framerate = sampled_filtered.framerate // factor

# Спектр
sampled_filtered_spectrum = sampled_filtered.make_spectrum()
sampled_filtered_spectrum.plot(high=10000)
plt.title('Спектр после дискретизации (с антиалиасингом)')
decorate(xlabel='Частота (Гц)', ylabel='Амплитуда')
plt.show()

# Прослушивание
sampled_filtered.make_audio()

**Комментарий:** Алиасинг устранен - спектр чистый без артефактов.

### Сравнение результатов

In [ ]:
# Сравнение спектров
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Исходный спектр
spectrum.plot(high=10000, ax=axes[0])
axes[0].set_title('Исходный спектр (44100 Гц)')
axes[0].set_xlabel('Частота (Гц)')
axes[0].set_ylabel('Амплитуда')
axes[0].grid(True, alpha=0.3)

# Без антиалиасинга
sampled_spectrum.plot(high=10000, ax=axes[1])
axes[1].axvline(cutoff, color='r', linestyle='--', alpha=0.5, label='Частота Найквиста')
axes[1].set_title('После дискретизации БЕЗ антиалиасинга (11025 Гц)')
axes[1].set_xlabel('Частота (Гц)')
axes[1].set_ylabel('Амплитуда')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# С антиалиасингом
sampled_filtered_spectrum.plot(high=10000, ax=axes[2])
axes[2].axvline(cutoff, color='r', linestyle='--', alpha=0.5, label='Частота Найквиста')
axes[2].set_title('После дискретизации С антиалиасингом (11025 Гц)')
axes[2].set_xlabel('Частота (Гц)')
axes[2].set_ylabel('Амплитуда')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Выводы:**
- Алиасинг возникает когда частота дискретизации недостаточна
- Частоты выше частоты Найквиста "отражаются" в низкочастотную область
- Антиалиасинговый фильтр применяется ДО дискретизации
- Частота среза фильтра должна соответствовать частоте Найквиста целевой частоты дискретизации
- После правильной фильтрации алиасинг полностью устраняется